# RAG Basics

This notebook teaches the smallest useful retrieval-augmented generation loop in the project. We stay close to the `src/` modules so that every step you study here maps directly to reusable code.

## Learning goals

- Understand what RAG adds to a plain question-answering system.
- See how document chunking changes what retrieval can find.
- Build a lightweight local vector index.
- Run a baseline QA pass and inspect its limitations.


## Concept explanation

Before we trust any notebook result, we should confirm which Python interpreter the kernel is using. In this project that should point at the uv-managed environment registered by `scripts/setup_uv_env.sh`.


In [ ]:
import sys
print(sys.executable)


This setup cell makes sure the notebook can import from `src/` whether it is launched from the repository root or directly from the `notebooks/` folder. Keeping this logic in one place prevents hidden import problems later in the tutorial.


In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists() and (PROJECT_ROOT.parent / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(PROJECT_ROOT)


### What is RAG

RAG combines two simple ideas: retrieve relevant context first, then generate an answer from that context. In this repository the baseline generator is intentionally simple so that you can see the retrieval effect clearly without hiding behind a large language model.


In [ ]:
import pandas as pd

from src.ingestion import build_demo_index, ingest_documents, load_documents
from src.workflow import run_baseline_rag

pd.set_option('display.max_colwidth', 120)
documents = load_documents()
pd.DataFrame(documents)[['doc_id', 'source']]


### What is document chunking

Retrievers rarely search whole documents well. They search smaller chunks so that each result is focused enough to match a question. Here we chunk each document into short windows of sentences and inspect the first few rows.

## Implementation


In [ ]:
chunks = ingest_documents(persist=False)
chunk_frame = pd.DataFrame(chunks)
chunk_frame[['doc_id', 'chunk_id', 'source', 'text']].head(8)


### What is embedding retrieval

The design spec calls for embeddings and vector retrieval. This implementation uses a local TF-IDF index as a lightweight stand-in for dense embeddings so the course runs anywhere. The retriever still behaves like a vector search component: it turns text into numeric features, scores similarity, and returns top-ranked chunks.


In [ ]:
retriever = build_demo_index(persist=False)
retrieval_query = 'What are the goals of the workspace policy refresh?'
retrieval_results = retriever.search(retrieval_query, top_k=4)
pd.DataFrame(retrieval_results)[['chunk_id', 'source', 'score', 'text']]


### Simple QA

Now we connect retrieval to answer generation. The baseline workflow normalizes the question, retrieves top chunks, and stitches together a short answer from the most relevant sentences.


In [ ]:
baseline_result = run_baseline_rag(
    'What are the main goals of the workspace policy refresh?',
    retriever=retriever,
)
print(baseline_result['final_answer'])
pd.DataFrame(baseline_result['citations'])


## Experiment

A good learning exercise is to try the same baseline on a question that really needs reasoning or tools. The pilot-window question requires reading dates and computing a difference, so it reveals where naive RAG starts to struggle.


In [ ]:
experiment_questions = [
    'How many days are in the pilot window?',
    'What are the first release priorities for the product assistant?',
]
experiment_rows = []
for question in experiment_questions:
    result = run_baseline_rag(question, retriever=retriever)
    experiment_rows.append(
        {
            'question': question,
            'final_status': result['final_status'],
            'answer': result['final_answer'],
            'sources': ', '.join(doc['source'] for doc in result['retrieved_docs'][:3]),
        }
    )
pd.DataFrame(experiment_rows)


## Result analysis

The baseline does well when the answer is already phrased in one or two nearby sentences. It is much weaker when the question needs decomposition, tool use, or a conservative abstention policy. Looking at the trace length and retrieved sources helps explain why.


In [ ]:
analysis_frame = pd.DataFrame(
    [
        {
            'query': baseline_result['user_query'],
            'retrieved_docs': len(baseline_result['retrieved_docs']),
            'trace_steps': len(baseline_result['trace']),
            'top_sources': ', '.join(doc['source'] for doc in baseline_result['retrieved_docs'][:2]),
        }
    ]
)
analysis_frame


## Takeaways

### Limitations of naive RAG

- Retrieval alone does not decide whether a question needs planning or tools.
- The baseline always answers; it does not know when to abstain.
- Grounding is implicit, not checked explicitly.

These gaps motivate the stateful agentic workflow in the next notebook.
